In [ ]:
import json
import requests
import pandas as pd
from pathlib import Path

from itertools import product


In [ ]:
## Project root path
pjpath = ''

# Hacky way of finding the project's root path. Do not rely on this, set your own pjpath!
for p in Path.cwd().parents:
    if p.stem == 'llms4mortality':
        pjpath = p
        break

print(f'> Project path is {pjpath}')

In [ ]:
# Relevant paths
mimicpath = pjpath / 'data/mimiciv'

In [ ]:
# Globals

SEED = 42

# Controls which data to load
samp_size = 5000
balanced_data = True
target_split = False  # Split set from loaded dataframe to generate summaries from. If None, we disregard inner splits and take entries directly from the main loaded dataframe

base_models = [
    #'llama3',
    #'hf.co/QuantFactory/Llama3-Med42-8B-GGUF:Q8_0',
    'hf.co/DevQuasar/google.medgemma-4b-it-GGUF:Q8_0',
]

# Path to json with system prompt
sysprompt_fpath = pjpath / 'ollama/sysprompts/sysprompt_course.json'


# Slice long input: Just keep up to max_words of each text
max_chars = 22000
subsamp_size = False  # 200, 100 Number of entries to test model with. Or False to disregard it

# This is the collection of columns that contains the relevant patient info
#   Values remap column name to an alternative and more readable name (might be useful if using LLMs)
pdc_remap = {
    'age': 'AGE',
    'gender': 'GENDER',
    'marital_status': 'MARITAL STATUS',
    'race': 'RACE',
    'diagnose_group_description': 'BROAD DIAGNOSIS',
    'diagnose_group_mortality': 'MORTALITY RISK',
    'insurance': 'INSURANCE',
    #'text': 'REPORT'
}

prepend_extended_patient_data = True    # If set True, preprends additional categorical data from pdf_remap to the beginning of the text entry for each patient

# Ollama hyperparams
n_ctx = 32
temp = 0.9
top_k = 90
top_p = 0.9


In [ ]:
# Load precomputed dataframe.
df = pd.read_csv(mimicpath / f'mimiciv_4_mortality_S{samp_size}{'_balanced' if balanced_data else ''}.csv.gz')

In [ ]:
# Load precomputed dataframe.
df = pd.read_csv(mimicpath / f'mimiciv_4_mortality_S{samp_size}{'_balanced' if balanced_data else ''}.csv.gz')

if target_split:
    print(f'>> (i) Parsing course only on entries from {target_split} split...')
    # Load precomputed splits
    with open(mimicpath / f'hadmid_splits_S{samp_size}{'_balanced' if balanced_data else ''}.json', 'r') as ifile:
        splits_hadmids = json.load(ifile)

    # Loads target split
    df = df.set_index('hadm_id').loc[splits_hadmids['test']]

# Do further subsamplig (do this just to speed up computations)
if subsamp_size:
    print(f'>> (!) Subsampling total entries to {subsamp_size}!')
    print(f'> Subsampling test data to {subsamp_size}...')
    df = df.sample(subsamp_size, random_state=SEED)

# Preprends additional data to the text entries
if prepend_extended_patient_data:
    print(f'>> (!) Preprending additional patient data to text!')
    df['text'] = df.apply(lambda x: ''.join([f'{p_cremap}: {str(x[p_cname]).replace('_', ' ')}\n' for p_cname, p_cremap in pdc_remap.items()]) + x['text'], axis=1)

# We are only interested in texts and mortality outcome within 30 days (and hadm_id ad index)
#df = df.set_index('hadm_id')['text'].to_frame()
df = df.set_index('hadm_id')[['text', 'delta_days_dod']]
df['dies_within_30d'] = df['delta_days_dod'].apply(lambda x: 'YES' if x > 0 and x <= 30 else 'NO')

In [ ]:
df.head()

In [ ]:
# Sets ollama instance and run

# First we load the system prompt configs from disk.
# These will be used to build the right system prompt for each experiment
with open(sysprompt_fpath, 'r') as ifile:
    sprompt_data = json.load(ifile)

# Getting related prompts for the specified input and output mode
sprompt = sprompt_data['prompt']

instance = 'http://localhost:11434/api/generate'
auth_cookie = ''
output_fpath = Path(f'{mimicpath}/precomputed_course')

# Creates output path in disk
output_fpath.mkdir(parents=True, exist_ok=True)

# Tiemout control
tout = 20
tout_count = 0

for base_model in base_models:
    print(f'> [BASE MODEL]: {base_model}')

    responses = {}
    i=1
    # Resolves responses disk path
    bmodel_name = base_model.replace('/', '').replace('.', '').replace(':', '_')
    output_id = f'course_S{str(samp_size)}{'_balanced' if balanced_data else ''}{'_sp' + target_split if target_split else ''}_{bmodel_name}_mc{str(max_chars)}{'_ss' + str(subsamp_size) if subsamp_size else ''}'    
    output_path = output_fpath / f'{output_id}.csv'

    # Tiemout control
    tout_path = output_fpath / f'_timedout_{output_id}.txt'   # Keeps track of timedout entries


    if output_path.is_file():
        # Loads existing file and assumes it contains the same data structure as the generated output (ie, is a compatible dataframe)
        print(f'>> (i) Target file already exists. Parsing contents and updating entries to process...')
        
        _df_existing_responses = pd.read_csv(output_path, index_col=0)
        assert list(_df_existing_responses.columns) == ['COURSE']

        precomputed_indices = _df_existing_responses.index
        print(f'>> (i) {len(precomputed_indices)} indices were found in precomputed results file and will be ommitted from current execution...')
        df = df.loc[list(set(df.index) - set(precomputed_indices))]


    else:
        # Initializes empty df where summaries will be saved online:
        pd.DataFrame(columns=['COURSE']).to_csv(output_path, mode='w', header=True)

        # Tiemout control
        if tout_path.is_file():
            open(tout_path, 'w').close()    # Flushes previous timeouts

    for index, row in df.iterrows():
        print(f'>> Processing row {i} out of {len(df)}', end='\r')

        # Get text and mortality outcome from entry
        text = row['text']
        mortality_outcome = row['dies_within_30d']

        # Truncate middle if resulting text is longer than max_chars
        if len(text) > max_chars:
            print(f'>> (!) Text exceeds the max char limit ({len(text)}) in entry {index}. Middle-truncating to {max_chars}...')
            text = text[:(max_chars//2)] + text[-(max_chars//2):]
            print(f'\t... Result truncate: {len(text)}')

        formatted_input = json.dumps({'REPORT': text, 'DIES': mortality_outcome})
        
        data = {'model': base_model,  # Explicit model to use
                'options': {
                                'num_ctx': n_ctx * 1024,
                                'temperature': temp, # 0?
                                'seed': SEED,
                                'top_k': top_k,
                                'top_p': top_p
                                },
                'keep-alive': 0,
                'system': sprompt,
                'prompt': formatted_input,
                'stream': False,  # Wait and return all the result at once
                'format': {  # Prognosis and mortality      
                'type': 'object',
                'properties': {
                    'COURSE': {
                        'type': 'string'
                    }
                },
                'required': [
                    'COURSE'
                ]
                }
            }
        # Prepares query
        data = json.dumps(data)
        cookies = {
            '_oauth2_proxy': auth_cookie}
        headers = {
            'Content-Type': 'application/x-www-form-urlencoded',
        }

        # Attempts call
        try:
            response = requests.post(instance, cookies=cookies, headers=headers, data=data, timeout=tout)
            response = json.loads(response.text)['response']
            df_response = pd.Series({index: json.loads(response)['COURSE']}).to_frame()
            df_response.to_csv(output_path, mode='a', header=False)
        except requests.Timeout:
            tout_count += 1
            print(f'ID {index} timed out (#{tout_count}), skipping and noting!')
            with open(tout_path, 'a+') as toutfile:
                toutfile.write(f'{str(index)}\n')

        except ValueError:
            # Warns about invalid response and skips
            print(f'(!) Invalid JSON format from request. Skipping ID {index}...')

        i+=1